# Ch 16. 位置エンコーディング — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: Sinusoidal PE の次元 0 と次元 32 の周波数を計算して比較せよ。

### 解答

偶数次元 $2i$ の角周波数は $\omega_i=10000^{-2i/d}$ である。$d=64$ では次元 0 は $i=0$ で 1、次元 32 は $i=16$ で $10^{-2}=0.01$ となり、後者は位置に対して 100 倍ゆっくり変化する。


In [ ]:
import numpy as np

d_model = 64
dimensions = np.array([0, 32])
frequencies = 10000.0 ** (-dimensions / d_model)
periods = 2 * np.pi / frequencies
assert np.allclose(frequencies, [1.0, 0.01])
print({int(d): {"angular_frequency": float(w), "period": round(float(p), 3)}
       for d, w, p in zip(dimensions, frequencies, periods)})


## 問題 2

**問題**: 学習可能な PE を 100 位置について学習し、位置 0〜50 と 51〜99 の違いを可視化せよ。

### 解答

学習可能な PE では位置間の滑らかさは自動的に保証されない。再現可能な小さな目的関数で 100 ベクトルを学習し、平均・分散・隣接差で 2 区間を比較し、見た目だけで結論しない。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Learn 100 position vectors against a fixed smooth target and compare the requested halves.
rng = np.random.default_rng(1602)
position = np.arange(100)[:, None]
frequency = np.arange(1, 9)[None, :] / 20
target = np.sin(position * frequency)
pe = rng.normal(scale=0.5, size=target.shape)
losses = []
for _ in range(120):
    error = pe - target
    losses.append(float(np.mean(error**2)))
    pe -= 0.2 * error
half_gap = float(np.linalg.norm(pe[:50].mean(0) - pe[50:].mean(0)))
adjacent = [float(np.linalg.norm(np.diff(block, axis=0), axis=1).mean()) for block in (pe[:51], pe[50:])]
fig, axis = plt.subplots(figsize=(6, 2.5)); axis.imshow(pe.T, aspect="auto", cmap="coolwarm"); plt.close(fig)
assert losses[-1] < 1e-12 and half_gap > 0
print({"initial_loss": round(losses[0], 5), "final_loss": losses[-1],
       "half_mean_gap": round(half_gap, 4), "adjacent_change": np.round(adjacent, 4).tolist()})


## 問題 3

**問題**: ALiBi のヘッド別傾き $m_h = 2^{-8h/H}$ を適用し、長い系列での効果をシミュレーションせよ。

### 解答

$m_h$ はヘッド番号とともに指数的に減少する。距離 $r$ のバイアスが $-m_hr$ なので、大きな傾きのヘッドは近距離を強く好み、小さな傾きのヘッドは長距離文脈を保つ。


In [ ]:
import numpy as np

heads, length = 8, 128
h = np.arange(1, heads + 1)
slopes = 2.0 ** (-8 * h / heads)
distance = np.arange(length)
bias = -slopes[:, None] * distance[None, :]
weights = np.exp(bias - bias.max(axis=1, keepdims=True)); weights /= weights.sum(axis=1, keepdims=True)
expected_distance = weights @ distance
assert np.all(np.diff(slopes) < 0) and np.all(np.diff(expected_distance) > 0)
print({"slopes": slopes.round(6).tolist(), "expected_attended_distance": expected_distance.round(2).tolist()})


## 問題 4

**問題**: RoPE の「相対距離のみに依存する」性質をさまざまな位置の組で検証せよ。

### 解答

2 次元回転ブロックでは $(R_mq)^T(R_nk)=q^TR_{n-m}k$ である。両位置を同じ量だけ移動しても内積は変わらない。以下で同じ相対距離を持つ 3 組の内積一致を確認する。


In [ ]:
import numpy as np

rng = np.random.default_rng(1604)
q, k = rng.normal(size=(2, 2))
theta = 0.17
def rotate(vector, position):
    angle = theta * position
    matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    return matrix @ vector
pairs = [(2, 9), (11, 18), (40, 47)]
scores = np.array([rotate(q, m) @ rotate(k, n) for m, n in pairs])
reference = q @ rotate(k, pairs[0][1] - pairs[0][0])
assert np.allclose(scores, reference, atol=1e-12)
print({"relative_distance": 7, "pair_scores": scores.round(10).tolist(), "max_error": float(np.max(np.abs(scores-reference)))})


## 問題 5

**問題**: Position Interpolation の式 $m \to m \cdot L_{train}/L_{test}$ を RoPE に適用し、外挿効果を説明せよ。

### 解答

位置に $s=L_{train}/L_{test}<1$ を掛けると、テスト時の最大回転角が学習範囲内に圧縮される。長さが 2 倍なら $s=1/2$ である。位置解像度は下がるが、未学習の過大な位相回転を避けられる。


In [ ]:
import numpy as np

train_length, test_length = 2048, 8192
scale = train_length / test_length
positions = np.array([0, test_length // 2, test_length - 1])
raw_angles = positions.astype(float)
interpolated_angles = positions * scale
assert interpolated_angles.max() < train_length and raw_angles.max() >= train_length
print({"scale": scale, "positions": positions.tolist(), "raw_angles": raw_angles.tolist(),
       "interpolated_angles": interpolated_angles.tolist(), "inside_training_range": True})
